Heart Disease Prediction Using Machine Learning	Perform load testing on the deployed model by simulating thousands of concurrent user requests, identifying bottlenecks in inference speed, and implementing performance optimizations to maintain stability.

In [1]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [4]:
from pathlib import Path
import joblib

BASE_DIR = Path.cwd()

print("Current Directory:", BASE_DIR)

# Search for the model file inside the project
model_files = list(
    BASE_DIR.rglob("heart_disease_model_v1.joblib")
)

print("\nModel Files Found:", len(model_files))

for file in model_files:
    print(file)

if not model_files:
    raise FileNotFoundError("Model file not found in project folder.")

MODEL_PATH = model_files[0]

print("\nSelected Model:", MODEL_PATH)
print("File Exists:", MODEL_PATH.exists())
print("File Size:", MODEL_PATH.stat().st_size, "bytes")

model = joblib.load(MODEL_PATH)

print("\nModel loaded successfully!")
print("Model Type:", type(model).__name__)

Current Directory: d:\brainybeam\new\Heart Disease Prediction

Model Files Found: 1
d:\brainybeam\new\Heart Disease Prediction\saved_models\heart_disease_model_v1.joblib

Selected Model: d:\brainybeam\new\Heart Disease Prediction\saved_models\heart_disease_model_v1.joblib
File Exists: True
File Size: 960 bytes

Model loaded successfully!
Model Type: Pipeline


In [5]:
features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

sample_data = pd.DataFrame([{
    "age": 55,
    "trestbps": 130,
    "chol": 240,
    "thalach": 150,
    "oldpeak": 1.5
}])

display(sample_data)

,age,trestbps,chol,thalach,oldpeak
0,55,130,240,150,1.5


In [6]:
def predict_heart_disease(data):

    processed_data = data[features].astype(float).to_numpy()

    prediction = model.predict(processed_data)

    probability = None

    if hasattr(model, "predict_proba"):
        probability = model.predict_proba(processed_data)[0].tolist()

    return {
        "prediction": int(prediction[0]),
        "probability": probability
    }

print(predict_heart_disease(sample_data))

{'prediction': 0, 'probability': [0.7034374429082094, 0.2965625570917906]}


In [7]:
latencies = []

for i in range(1000):

    start = time.perf_counter()

    prediction = predict_heart_disease(sample_data)

    end = time.perf_counter()

    latencies.append((end - start) * 1000)

print("Total Predictions:", len(latencies))
print("Average Latency:", round(np.mean(latencies), 4), "ms")
print("Minimum Latency:", round(np.min(latencies), 4), "ms")
print("Maximum Latency:", round(np.max(latencies), 4), "ms")
print("95th Percentile:", round(np.percentile(latencies, 95), 4), "ms")

Total Predictions: 1000
Average Latency: 4.5455 ms
Minimum Latency: 2.0756 ms
Maximum Latency: 22.2696 ms
95th Percentile: 8.0735 ms


In [9]:
!pip install fastapi uvicorn locust requests 

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 1.4 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.5 MB 1.5 MB/s eta 0:00:01
   ---------------------------------- ----- 1.3/1.5 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.7 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.7 MB 2.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.6/1.7 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.3 MB 5.0 MB/s eta 0:00:01
   -------------------------------- ------- 1.8/2.3 M

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Jupyter Notebook working directory
BASE_DIR = Path.cwd()

# Find the saved model
MODEL_PATH = next(
    BASE_DIR.rglob("heart_disease_model_v1.joblib")
)

model = joblib.load(MODEL_PATH)

app = FastAPI(title="Heart Disease Prediction API")


class PatientData(BaseModel):
    age: float = Field(ge=1, le=120)
    trestbps: float = Field(gt=0)
    chol: float = Field(gt=0)
    thalach: float = Field(gt=0)
    oldpeak: float


@app.get("/")
def home():
    return {"message": "Heart Disease Prediction API"}


@app.post("/predict")
def predict(data: PatientData):

    values = pd.DataFrame([{
        "age": data.age,
        "trestbps": data.trestbps,
        "chol": data.chol,
        "thalach": data.thalach,
        "oldpeak": data.oldpeak
    }])

    prediction = model.predict(values)

    return {
        "prediction": int(prediction[0])
    }


print("FastAPI application created successfully!")
print("Model:", type(model).__name__)
print("Model Path:", MODEL_PATH)

FastAPI application created successfully!
Model: Pipeline
Model Path: d:\brainybeam\new\Heart Disease Prediction\saved_models\heart_disease_model_v1.joblib


# Running Heart Disease Prediction API on Localhost

### Step 1: Save API code
Save your FastAPI code in a Python file named `api.py` inside the `Heart Disease Prediction` folder.

### Step 2: Install required libraries

Open the VS Code terminal:

```bash
pip install fastapi uvicorn pandas numpy scikit-learn joblib
```

### Step 3: Start the local server

Run:

```bash
uvicorn api:app --host 127.0.0.1 --port 8000
```

### Step 4: Open API documentation

Open this URL in your browser:

http://127.0.0.1:8000/docs

### Step 5: Test prediction

1. Expand the `POST /predict` endpoint.
2. Click **Try it out**.
3. Enter sample input:

```json
{
  "age": 55,
  "trestbps": 130,
  "chol": 240,
  "thalach": 150,
  "oldpeak": 1.5
}
```

4. Click **Execute**.
5. Check the prediction response.

### Step 6: Stop the server

Press `CTRL + C` in the terminal.

### Expected Result

- FastAPI server runs locally.
- API documentation opens successfully.
- Patient input is accepted.
- The trained model generates a prediction.
- API response is returned in JSON format.

### Conclusion

The Heart Disease Prediction model was integrated with FastAPI and executed on a local server. The API was tested through Swagger UI to verify patient input handling and prediction responses.

Note: Localhost testing does not mean the API is publicly deployed.
